# 03 - Feature Engineering

Input: `data/house_prices_no_outliers.csv` from `02_eda.ipynb`. Goal: bucket the high-cardinality `location` column and produce a feature matrix ready for step 5 modeling.

In [1]:
import pandas as pd

df = pd.read_csv('../data/house_prices_no_outliers.csv')
df.shape

(12540, 9)

## Dropping `area_type` and `availability`

The planned `/predict` endpoint (step 7) only collects `location`, `size`, `total_sqft`, `bath`, `balcony` from the user, so `area_type` and `availability` can't be model inputs, the API will never have them at inference time.

Worth being explicit that this is a real trade-off, not a free drop: `area_type` does carry signal (checked below). Dropping it is a scope decision (keep the form to 5 fields) more than a data quality one.

In [2]:
df['price_per_sqft'] = df['price'] * 100000 / df['total_sqft']
df.groupby('area_type')['price_per_sqft'].median().sort_values(ascending=False)

area_type
Plot  Area              8822.489392
Carpet  Area            5909.090909
Super built-up  Area    5111.946533
Built-up  Area          5066.251316
Name: price_per_sqft, dtype: float64

Plot Area listings run about 73% higher per sqft than Built-up or Super built-up (8,822 vs ~5,100). That's a real gap, so this is signal left on the table, not noise. Noting it here in case a future version of the form adds an area-type field back in.

`availability` is 80 values, but 80% of rows are `Ready To Move` and the rest are specific possession dates (`18-Dec`, `19-Mar`, etc). Possession date isn't really a size/location/quality signal the way the other features are, and it's not in the planned form either, so it's dropped without the same signal check.

In [3]:
df = df.drop(columns=['area_type', 'availability', 'price_per_sqft'])
df.columns.tolist()

['location', 'bath', 'balcony', 'price', 'total_sqft', 'bhk']

## Bucketing `location`

From the EDA: 1,304 unique locations, 81% with 10 or fewer listings. One-hot encoding all of them gives hundreds of columns that each fire on a handful of rows. Sweeping a few cutoffs to see the tradeoff between category count and how much data gets folded into `other`:

In [4]:
vc = df['location'].value_counts()
for cutoff in [5, 10, 15, 20, 30]:
    n_kept = (vc > cutoff).sum()
    rows_other = df['location'].isin(vc[vc <= cutoff].index).sum()
    print(f'cutoff {cutoff:>2}: {n_kept:>3} locations kept, {rows_other:>5} rows ({rows_other/len(df):.1%}) folded to other')

cutoff  5: 362 locations kept,  1702 rows (13.6%) folded to other
cutoff 10: 223 locations kept,  2726 rows (21.7%) folded to other
cutoff 15: 167 locations kept,  3442 rows (27.4%) folded to other
cutoff 20: 135 locations kept,  4004 rows (31.9%) folded to other
cutoff 30:  94 locations kept,  5027 rows (40.1%) folded to other


Going with cutoff 10 (a location needs more than 10 listings to keep its own column): 223 locations kept, 21.7% of rows fold into `other`. Lower cutoffs (5) keep more categories but a lot of them are still thin (6-10 listings isn't enough to learn a reliable location effect). Higher cutoffs (20+) start folding away well-represented, genuinely distinct areas, over 30% of the data would share one `other` bucket.

In [5]:
KEEP_LOCATIONS = sorted(vc[vc > 10].index.tolist())
len(KEEP_LOCATIONS)

223

In [6]:
df['location'] = df['location'].apply(lambda loc: loc if loc in KEEP_LOCATIONS else 'other')
df['location'].nunique()

224

## One-hot encoding, without the Titanic pitfall

The Titanic README flags `pd.get_dummies(drop_first=True)` as broken for single-row inference: which category it drops depends on what's present in that specific call, so a live `/predict` call with one row doesn't get the same columns as training did.

Same fix here: build the dummy columns from a fixed, saved category list (`KEEP_LOCATIONS` + `other`), not by re-running `get_dummies` on whatever happens to be in the DataFrame at the time.

In [7]:
LOCATION_COLUMNS = [f'location_{loc}' for loc in KEEP_LOCATIONS + ['other']]

for col, loc in zip(LOCATION_COLUMNS, KEEP_LOCATIONS + ['other']):
    df[col] = (df['location'] == loc).astype(int)

df = df.drop(columns=['location'])
df.shape

C:\Users\Abuu Safwaan\AppData\Local\Temp\ipykernel_3084\1076401184.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = (df['location'] == loc).astype(int)
C:\Users\Abuu Safwaan\AppData\Local\Temp\ipykernel_3084\1076401184.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = (df['location'] == loc).astype(int)
C:\Users\Abuu Safwaan\AppData\Local\Temp\ipykernel_3084\1076401184.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

(12540, 229)

This also means `KEEP_LOCATIONS` has to ship with the model, step 6 (`src/preprocess.py`) needs it to build the same columns at inference time from a single-row request. Saving it now so it's not recomputed from scratch later, which would silently drift if the training data ever changes.

In [8]:
import json

with open('../data/keep_locations.json', 'w') as f:
    json.dump(KEEP_LOCATIONS, f)

print(f'saved {len(KEEP_LOCATIONS)} locations')

saved 223 locations


## Final feature check

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12540 entries, 0 to 12539
Columns: 229 entries, bath to location_other
dtypes: float64(5), int64(224)
memory usage: 21.9 MB


In [10]:
df.to_csv('../data/features.csv', index=False)
df.shape

(12540, 229)

Output: `data/features.csv` (price + bath, balcony, total_sqft, bhk + one-hot location columns), and `data/keep_locations.json` (the fixed category list step 6 needs). Ready for step 5, model training and comparison.